# Нейронные сети. Основы

## Реализация перцептрона

Перцептрон - это модель, предложенная Френком Розенблаттом в 1957 году и являющаяся прообразом современных нейронных сетей. По своей сути она представляет из себя значительно упрощенную схему восприятия информации мозгом. Целью этого практического задания будет реализация собственной модели перцептрона. Давайте разберем схему работы этого алгоритма в деталях.

* Мы работаем с тренировочной выборкой $S = \{(x_i, y_i)| i \in \{1,...,m\} \}$
* Инициализируем веса $\omega^{(0)} \leftarrow 0$ нулевым вектором.
* Инициализирует bias параметр $b = 0$.
* В начальный момент времени номер шага $t=0$.
* Задаем learning rate $\eta > 0$.
* Пока значение $t < t_{\max}$
    * случайно выбираем объект из тренировочной выборки $(x_i, y_i) \in S$.
    * если выполняется условие $y_i (\langle \omega^{(t)}, x_i\rangle + b) \leq 0$ тогда
        * $b^{(t+1)} \leftarrow b^{(t)} + \eta \times y_i$.
        * $\omega^{(t+1)} \leftarrow \omega^{(t)} + \eta \times y_i \times x_i$.
    * далее, обновляем $t \leftarrow t+1$.

Таким образом, финальное значение ветора весов $\omega$ и bias параметра $b$ позволяют классифицировать новый объект $x$. Если $(\langle \omega, x \rangle + b) \geq 0$, то мы относим объект к классу $+1$, в противном случае мы относим объект к классу $-1$.

Для начала загрузим датасет для задачи классификации цветков Ириса с помощь функции `load_iris` из `sklearn.datasets`. Давайте подготовим данные для задачи бинарной классификации. Для этого выберем первые 100 элементов из данного набора данных. Так же преобразуем класс $0$ в класс $-1$.

In [1]:
import numpy as np
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)
X, y = X[:100], y[:100]
num_features = X.shape[1]
y = np.array([1 if y_i == 1 else -1 for y_i in y])

Реализуйте алгоритм перцептрона приведенный выше. Для выборки случайного объекта из тренировочного датасета по индексу используйте функцию `randint` из модуля `random` с параметрами 0 и n, где n - это размер тренировочно выборки. Перед запуском итераций алгоритма установите `random.seed(42)`. Вы можете реализовать перцептрон в качестве класса с интерфейсом, похожим на интерфейсы моделей из `scikit-learn`. Для этого достаточно реализовать функцию `fit` и `predict` для решения этого практического задания. Однако, ваша реализация может отличаться. Необходимое требование - это использование генератора случайных чисел, описанного выше.

### *РЕШЕНИЕ*

In [95]:
import random


class Perceptron:
    import numpy as np

    def __init__(self, num_features, learning_rate, tmax, random_state=42):
        self.weights = [0, 0, 0, 0]  # Инициализируем веса 𝜔(0)←0 нулевым вектором
        self.bias = 0  # Инициализирует bias параметр 𝑏=0
        self.t = 0  # В начальный момент времени номер шага 𝑡=0
        self.num_features = num_features
        self.learning_rate = learning_rate  # Задаем learning rate 𝜂>0
        assert learning_rate > 0, 'learning_rate should be greatger than 0!'
        self.tmax = tmax
        self.random_state = random_state

    def fit(self, X, y=None):  # Мы работаем с тренировочной выборкой 𝑆={(𝑥𝑖,𝑦𝑖)|𝑖∈{1,...,𝑚}}
        random.seed(self.random_state)
        while self.t < self.tmax:  # Пока значение 𝑡<𝑡max
            rand = random.randint(0, len(X)-1)  # случайно выбираем объект из тренировочной выборки (𝑥𝑖,𝑦𝑖)∈𝑆
#             if (np.dot(self.weights, X[rand]) + self.bias) <= 0:  # если выполняется условие 𝑦𝑖(⟨𝜔(𝑡),𝑥𝑖⟩+𝑏)≤0
            if np.dot(y[rand],(np.dot(self.weights, X[rand]) + self.bias)) <= 0:  # если выполняется условие 𝑦𝑖(⟨𝜔(𝑡),𝑥𝑖⟩+𝑏)≤0
                self.bias = self.bias + self.learning_rate * y[rand]  # 𝑏(𝑡+1)←𝑏(𝑡)+𝜂×𝑦𝑖
                self.weights = np.add(self.weights,
                                      np.multiply(X[rand], y[rand]*self.learning_rate))  # 𝜔(𝑡+1)←𝜔(𝑡)+𝜂×𝑦𝑖×𝑥𝑖
            self.t += 1
        del (self.t, self.tmax, self.learning_rate,
             self.random_state, self.num_features)
        return self

    def predict(self, X):
        y_pred = []  # будет содержать наши предсказания
        for obj in X:  # перебор всех значений X
            if (np.dot(self.weights, obj) + self.bias) >= 0:  # (⟨𝜔,𝑥⟩+𝑏)≥0
                pred = 1
            else:
                pred = -1
            y_pred.append(pred)  # добавляем предсказанный класс в список
        return y_pred

    def get_params(self, deep=True):
        params = self.__dict__
        for par in params:
            print(par, '=', params[par])
        print('---\n')

Следующим шагом является проверка нашей модели. Случайно разделите выборку на тренировочный и тестовый датасет, используя функцию `tran_test_split` с параметрами `test_size=0.25` и `random_state=10`. Запустите обучение модели с параметрами $\eta=0.1$ и $t_{\max}=40$. Оцените качество на тестовой выборке и запишите результат в переменную `score` с точность до двух знаков после запятой, используя метрику `accuracy`. Это значение и будет являться ответом на это практическое задание.

### *РЕШЕНИЕ*

In [96]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=10)
model = Perceptron(num_features, 0.1, 40)
model.get_params()
model.fit(X_train, y_train)
model.get_params()
y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)
score = round(accuracy_score(y_test, y_pred), 2)

print("score {0:.2f}".format(score))

weights = [0, 0, 0, 0]
bias = 0
t = 0
num_features = 4
learning_rate = 0.1
tmax = 40
random_state = 42
---

weights = [-0.2  -0.89  1.26  0.51]
bias = -0.1
---

score 1.00


# Строка с ответами

In [97]:
print("score {0:.2f}".format(score))

score 1.00
